In [ ]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "false"

import jax
import jax.numpy as jnp
import optax

print("Backend:", jax.default_backend())
print("Devices:", jax.devices())
print("X64 enabled:", jax.config.read("jax_enable_x64"))

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MinMaxScaler
import pickle

from experiments_params import generate_models_in_dictionary, generate_multiclass_datasets
from iqc import IQC
# from iqc_de import next_power_of_two


data_bases = generate_multiclass_datasets()
results = {}
name_of_file = "results_multiclass_PSO_with_regularization.pkl"

try:
    with open(name_of_file, "rb") as file:
        results = pickle.load(file)
except FileNotFoundError:
    print("File not found, starting with an empty dictionary.")

for random_state in [40, 80, 120, 160, 200, 240, 280, 320, 360, 400]:
    for dataset_name, (X, y) in data_bases.items():
        print("Dataset", dataset_name, "random state", random_state)
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            stratify=y,
            random_state=random_state,
        )
        scaler = MinMaxScaler(feature_range=(0, 1))
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        models = generate_models_in_dictionary(X.shape[1])

        for model_name, model_info in models.items():
            optimizer_name = "pso"
            result_key = (dataset_name, model_name, optimizer_name, random_state)
            if result_key in results:
                print(f"Skipping {result_key}: already computed.")
                continue

            if model_name.startswith("iqc_multidimensional"):
                n_environment = int(model_name.split("_")[2])
            # elif model_name.startswith("iqc_de"):
            #     n_environment = next_power_of_two(X.shape[1])
            else:
                n_environment = 2

            binary_iqc = IQC(
                model_name=model_name,
                iqc=model_info["iqc"],
                number_of_params=model_info["number_of_params"],
                method=optimizer_name,
                n_particles=30,
                max_steps=500,
                verbose=False,
                random_state=random_state,
                N_e=n_environment,
            )
            multiclass_iqc = OneVsRestClassifier(binary_iqc, n_jobs=1)
            multiclass_iqc.fit(X_train_scaled, y_train)
            y_pred = multiclass_iqc.predict(X_test_scaled)

            best_params = tuple(
                estimator.params_ for estimator in multiclass_iqc.estimators_
            )
            results[result_key] = {
                "accuracy": (y_pred == y_test).mean(),
                "f1_score": f1_score(y_test, y_pred, average="macro"),
                "recall": recall_score(y_test, y_pred, average="macro"),
                "precision": precision_score(y_test, y_pred, average="macro"),
                "best_params": best_params,
            }

            with open(name_of_file, "wb") as file:
                pickle.dump(results, file)